# 02 — Onset Detection (Simplified Liebmann-Marengo)

**Purpose:** Detect onset (start) of wet season untuk setiap tahun 1995-2025 di Kebumen.

## Method: Simplified Liebmann-Marengo

Onset detection rule:
1. Cari hari pertama setelah 1 Agustus dimana akumulasi 5-day rolling rainfall ≥ 25mm
2. DAN dalam 30 hari berikutnya, gak ada dry spell (rainfall < 1mm) ≥ 7 hari berturut-turut

**Catatan:** Versi original Liebmann-Marengo (2001) pakai threshold-based approach yang lebih kompleks dengan accumulated anomaly. Kita pakai versi simplified yang mempertahankan core logic tapi lebih cepat dijalankan dan diinterpretasi.

## Inputs
- `data/processed/rainfall_daily_clean.csv` (output dari notebook 01)

## Outputs
- `data/processed/onset_dates.csv` — onset date + DOY per year
- `data/processed/onset_detection_log.csv` — detail rolling sum & dry streak per attempt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_PROCESSED = Path('../data/processed')

# Load cleaned rainfall
rainfall = pd.read_csv(DATA_PROCESSED / 'rainfall_daily_clean.csv')
rainfall['date'] = pd.to_datetime(rainfall['date'])
print(f"Loaded rainfall: {rainfall.shape}")
print(f"Date range: {rainfall['date'].min().date()} → {rainfall['date'].max().date()}")

## Onset Detection Function

In [ ]:
def detect_onset(rainfall_df, year, 
                 search_start_month=8, 
                 search_end_month=12,
                 rolling_window=5,
                 rainfall_threshold_mm=25,
                 post_check_days=30,
                 max_dry_spell_days=7,
                 dry_threshold_mm=1):
    """
    Simplified Liebmann-Marengo onset detection.
    
    Returns:
        dict with onset_date, onset_doy, rolling_sum_at_onset, max_dry_streak_post
        atau None kalau tidak ditemukan
    """
    # Filter season window
    start = pd.Timestamp(f'{year}-{search_start_month:02d}-01')
    end = pd.Timestamp(f'{year}-{search_end_month:02d}-31') if search_end_month != 12 else pd.Timestamp(f'{year}-12-31')
    season = rainfall_df[(rainfall_df['date'] >= start) & (rainfall_df['date'] <= end)].copy()
    season = season.sort_values('date').reset_index(drop=True)
    
    if len(season) < rolling_window + post_check_days:
        return None
    
    # Compute rolling sum
    season['roll_sum'] = season['rainfall_mm'].rolling(rolling_window).sum()
    
    # Search for onset candidate
    for i in range(rolling_window - 1, len(season) - post_check_days):
        if season.iloc[i]['roll_sum'] >= rainfall_threshold_mm:
            # Check next post_check_days for dry spell
            post_window = season.iloc[i+1:i+1+post_check_days]['rainfall_mm'].values
            
            dry_streak = 0
            max_dry = 0
            for r in post_window:
                if r < dry_threshold_mm:
                    dry_streak += 1
                    max_dry = max(max_dry, dry_streak)
                else:
                    dry_streak = 0
            
            if max_dry < max_dry_spell_days:
                onset_date = season.iloc[i]['date']
                return {
                    'year': year,
                    'onset_date': onset_date,
                    'onset_doy': onset_date.dayofyear,
                    'rolling_sum_at_onset': season.iloc[i]['roll_sum'],
                    'max_dry_streak_post': max_dry
                }
    
    return None

# Test on single year
test_result = detect_onset(rainfall, 2020)
print("Test detection 2020:")
print(test_result)

In [ ]:
# Apply for all years 1995-2025
results = []
failed_years = []

for year in range(1995, 2026):
    result = detect_onset(rainfall, year)
    if result:
        results.append(result)
    else:
        failed_years.append(year)
        print(f"⚠️ No onset detected for {year}")

onset_df = pd.DataFrame(results)
print(f"\n✅ Detected onset for {len(onset_df)}/31 years")
print(f"Failed years: {failed_years}")
onset_df.head(10)

In [ ]:
# Sanity check stats
print("Onset DOY statistics:")
print(onset_df['onset_doy'].describe())
print(f"\nOnset date range: {onset_df['onset_date'].min().date()} → {onset_df['onset_date'].max().date()}")
print(f"\nEarliest onset: {onset_df.loc[onset_df['onset_doy'].idxmin()][['year', 'onset_date', 'onset_doy']].to_dict()}")
print(f"Latest onset: {onset_df.loc[onset_df['onset_doy'].idxmax()][['year', 'onset_date', 'onset_doy']].to_dict()}")

In [ ]:
# Visualize: onset DOY over time
fig, ax = plt.subplots(figsize=(14, 5))

# Plot points + line
ax.plot(onset_df['year'], onset_df['onset_doy'], 'o-', color='#7A8450', markersize=8, linewidth=1.5)

# Reference: traditional Pranata Mangsa onset (~22 Oct = DOY 295)
ax.axhline(295, color='#C8553D', linestyle='--', linewidth=2, label='Pranata Mangsa traditional (Oct 22, DOY 295)')

# Highlight extreme El Niño years
el_nino_years = [1997, 2015, 1982, 1991, 2009, 2023]  # known strong El Niños
for ey in el_nino_years:
    if ey in onset_df['year'].values:
        doy = onset_df[onset_df['year'] == ey]['onset_doy'].values[0]
        ax.annotate(f'{ey}\n(El Niño)', xy=(ey, doy), xytext=(ey, doy+8),
                    fontsize=8, ha='center', color='red',
                    arrowprops=dict(arrowstyle='->', color='red', alpha=0.5))

ax.set_xlabel('Year')
ax.set_ylabel('Onset Day of Year')
ax.set_title('Wet Season Onset Detection — Kebumen 1995-2025')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/onset_timeseries.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Compute shift from traditional baseline
TRADITIONAL_ONSET_DOY = 295  # ~22 October
onset_df['shift_days'] = onset_df['onset_doy'] - TRADITIONAL_ONSET_DOY

print("Shift days statistics:")
print(onset_df['shift_days'].describe())

# Histogram of shifts
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(onset_df['shift_days'], bins=15, color='#5B8AA6', edgecolor='black')
ax.axvline(0, color='black', linestyle='--', label='Traditional baseline')
ax.axvline(14, color='red', linestyle='--', label='Alert threshold (+14 days)')
ax.axvline(-14, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Shift from traditional onset (days)')
ax.set_ylabel('Count of years')
ax.set_title('Distribution of Onset Shifts — Kebumen 1995-2025')
ax.legend()
plt.tight_layout()
plt.savefig('../docs/onset_shift_histogram.png', dpi=100, bbox_inches='tight')
plt.show()

# Years with extreme shift
extreme_shifts = onset_df[abs(onset_df['shift_days']) > 14].sort_values('shift_days', key=abs, ascending=False)
print(f"\nYears with shift > 14 days ({len(extreme_shifts)} years):")
print(extreme_shifts[['year', 'onset_date', 'onset_doy', 'shift_days']].to_string(index=False))

In [ ]:
# Save outputs
onset_df['onset_date'] = onset_df['onset_date'].dt.strftime('%Y-%m-%d')
onset_df.to_csv(DATA_PROCESSED / 'onset_dates.csv', index=False)
print(f"✅ Saved: {DATA_PROCESSED / 'onset_dates.csv'}")
print(f"   Shape: {onset_df.shape}")
onset_df.head()

## Validation Notes

**Expected behavior (sanity check):**
- Strong El Niño years (1997, 2015, 2023) → onset delayed (positive shift, +10 to +30 days)
- Strong La Niña years (1998, 2010, 2020) → onset earlier (negative shift, -5 to -15 days)
- Neutral years → shift around ±5 days

**Kalau sanity check failed** (misalnya El Niño year malah onset lebih awal):
1. Check apakah CHIRPS data untuk tahun tersebut complete
2. Adjust threshold parameters (rainfall_threshold_mm, max_dry_spell_days)
3. Document deviation di limitations section README

**Next step:** Run `03_train_model.ipynb` untuk feature engineering + Random Forest training.